# B07A — Pycol: revisi pemilihan subset berdasarkan label pasien

Notebook **lanjutan** B07, khusus memperbaiki analisis sensitivitas subset. Memakai cache fitur ResNet50 B07 `453add0862207787`. **Tidak melatih model, tidak mengekstrak fitur, tidak membutuhkan GPU, dan tidak membuka citra/test.**

Jalankan Cell 1–6 di Colab (CPU cukup). Drive harus masih memiliki folder hasil B07 dan CSV train B04. Semua hasil baru berada di `b07a_pycol_label_aware_v1`, terpisah dari B07.

**Perubahan:** pilih pasien untuk memenuhi kuota negatif/positif terlebih dahulu, kemudian pilih satu citra dengan label yang sesuai. Pasien dengan kedua label dialokasikan ke satu kelas dalam satu subset. Label citra tidak diubah.

Jumlah seimbang maksimum K memenuhi `K <= pasien eligible negatif`, `K <= pasien eligible positif`, dan `2K <= seluruh pasien`. K bersama adalah minimum kapasitas keempat dataset, dengan cap 250. Untuk laporan B07 saat ini, K=21 sehingga ada **42 citra dari 42 pasien** per subset.

Distribusi pengambilan sampel didefinisikan secara eksplisit: acak pasangan jumlah pasien campuran yang layak secara seragam, lalu acak identitas pasien tanpa penggantian, lalu satu citra sesuai label. Ini bukan sampling seragam atas semua subset yang mungkin. Pada INbreast semua 21 pasien eligible positif terpilih setiap pengulangan.

**Batas interpretasi:** 20 pengulangan adalah sensitivitas pemilihan subset, bukan 20 eksperimen training independen, bukan bootstrap CI. Ukuran, prevalensi, dan komposisi pasien berbeda dari full train. Perbedaan B07–B07A mencakup perubahan sampler **dan** ukuran subset; bukan efek murni satu faktor. Kompleksitas tetap bergantung pada fitur ImageNet, normalisasi HEOM per subset, cohort, dan target label. Tidak ada ranking agregat atau uji signifikansi.

Patient_id mengikuti grouping yang disimpan pada B07; notebook ini tidak mengaudit ulang identitas klinis. Metadata/caveat sumber dipertahankan. INbreast memakai BI-RADS 1/2 vs 5/6, MIAS abnormal-only.

Referensi Pycol: https://doi.org/10.1016/j.neucom.2025.130311 ; paket 1.0.4: https://pypi.org/project/pycol-complexity/1.0.4/

## Cell 1 — Persiapan CPU dan lokasi
Tidak perlu menjalankan B07 ulang. Ubah `RESEARCH` hanya jika lokasi Drive berbeda.

In [ ]:
import sys,subprocess,importlib.metadata as metadata
subprocess.check_call([sys.executable,'-m','pip','install','-q','pycol-complexity==1.0.4'])
import numpy as np,pandas as pd
import json,hashlib,zipfile,os,inspect,platform
from pathlib import Path
from datetime import datetime,timezone
import matplotlib.pyplot as plt
from pycol_complexity import complexity as pycol_module
from google.colab import drive,files
from IPython.display import display
drive.mount('/content/drive')
RESEARCH=Path('/content/drive/MyDrive/Research')
SOURCE_ID='453add0862207787'
SOURCE=RESEARCH/'outputs/benchmark_4datasets/b07_pycol_v1'/SOURCE_ID
REGISTRY_PATH=RESEARCH/'outputs/benchmark_4datasets/b04_preprocessing_v1/1dcb7ddadfe6/registry.json'
OUTPUT_ROOT=RESEARCH/'outputs/benchmark_4datasets/b07a_pycol_label_aware_v1'
DATASETS=['cbis_ddsm','cmmd','inbreast','mias']
REPEATS=20
SUBSET_SEED=42
CAP_PER_CLASS=250
DOWNLOAD_REPORT=True
assert REPEATS>=2 and CAP_PER_CLASS>=2
if not (SOURCE/'protocol.json').exists(): raise FileNotFoundError(f'Folder B07 tidak ditemukan: {SOURCE}')


## Cell 2 — Mesin Pycol yang sama dan sampler baru
Definisi metrik serta normalisasi sama dengan B07. Hanya desain subset yang diganti.

In [ ]:
ENGINE_SOURCE='import hashlib, json, os, gc, warnings\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy.spatial.distance import cdist\nfrom pycol_complexity.complexity import Complexity\n\ndef file_sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(8*1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\n\ndef object_sha(obj):\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,allow_nan=False).encode()).hexdigest()\n\ndef save_json(path,obj):\n    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_suffix(path.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)); os.replace(tmp,path)\n\ndef read_json(path): return json.loads(Path(path).read_text())\n\nclass FastNumericComplexity(Complexity):\n    """Pycol 1.0.4, identical numeric HEOM via vectorized scipy cdist.\n    No categorical/missing input. Metric methods remain upstream Pycol.\n    Per-cohort range normalization is upstream behavior, not a shared scaler.\n    """\n    def _Complexity__calculate_distance_matrix(self,X,distance_func=\'default\'):\n        X=np.asarray(X,dtype=np.float64)\n        if distance_func not in (\'default\',\'HEOM\') or not np.isfinite(X).all():\n            raise ValueError(\'Only finite numeric HEOM input is supported\')\n        span=np.ptp(X,axis=0); span[span==0]=1.\n        Z=(X-X.min(axis=0))/span\n        return cdist(Z,Z),cdist(X,X)\n\ndef make_complexity(X,y):\n    X=np.asarray(X,dtype=np.float64); y=np.asarray(y,dtype=int)\n    if X.ndim!=2 or len(X)!=len(y) or not np.isfinite(X).all(): raise ValueError(\'Invalid features\')\n    if set(y)!={0,1} or min(np.bincount(y))<2: raise ValueError(\'Need >=2 images in each class\')\n    return FastNumericComplexity(file_type=\'array\',dataset={\'X\':X,\'y\':y,\'meta\':[0]*X.shape[1]})\n\ndef self_test():\n    rng=np.random.default_rng(917)\n    X=rng.normal(size=(14,7)); X[:7]+=1.5; X[:,6]=2.\n    y=np.array([0]*7+[1]*7)\n    a=Complexity(file_type=\'array\',dataset={\'X\':X,\'y\':y,\'meta\':[0]*7})\n    b=make_complexity(X,y)\n    np.testing.assert_allclose(a.dist_matrix,b.dist_matrix,rtol=1e-12,atol=1e-12)\n    np.testing.assert_allclose(a.unnorm_dist_matrix,b.unnorm_dist_matrix,rtol=1e-12,atol=1e-12)\n    for method in (\'N1\',\'N2\',\'N3\'):\n        for imb in (False,True):\n            np.testing.assert_allclose(getattr(a,method)(imb=imb),getattr(b,method)(imb=imb),rtol=1e-12,atol=1e-12)\n    with warnings.catch_warnings():\n        warnings.simplefilter(\'ignore\',RuntimeWarning)\n        np.testing.assert_allclose(a.F1(),b.F1(),rtol=1e-12,atol=1e-12)\n    return {\'numeric_HEOM_matches_upstream\':True,\'N1_N2_N3_F1_match\':True,\'test_seed\':917}\n\ndef measure(X,frame,dataset,scenario=\'full_train\',repeat=-1,patient_checks=True):\n    """Long-format metrics; extra patient-excluded quantities are NOT native Pycol."""\n    y=frame.label.to_numpy(dtype=int); groups=frame.patient_id.to_numpy(str)\n    c=make_complexity(X,y); rows=[]\n    def add(metric,scope,value,implementation=\'pycol_1.0.4\',status=\'ok\'):\n        value=float(value)\n        rows.append(dict(dataset=dataset,scenario=scenario,repeat=int(repeat),metric=metric,\n            scope=str(scope),value=value if np.isfinite(value) else np.nan,status=status,\n            implementation=implementation,images=len(y),patients=len(set(groups)),\n            negative=int((y==0).sum()),positive=int((y==1).sum())))\n    off=c.dist_matrix[np.triu_indices(len(y),1)]\n    cross=c.dist_matrix[y==0][:,y==1]\n    for metric in (\'N1\',\'N2\',\'N3\'):\n        bad=(metric==\'N1\' and (off<=1e-8).any()) or (metric==\'N2\' and (cross==0).any())\n        if bad:\n            reason=\'undefined: zero/near-zero MST edges\' if metric==\'N1\' else \'undefined: cross-class zero distance\'\n            for scope in (\'overall\',\'class_0\',\'class_1\',\'macro\'): add(metric,scope,np.nan,status=reason)\n            continue\n        overall=getattr(c,metric)(); per=np.asarray(getattr(c,metric)(imb=True))\n        add(metric,\'overall\',overall)\n        for label,value in zip(c.classes,per): add(metric,f\'class_{label}\',value)\n        add(metric,\'macro\',per.mean(),implementation=\'mean_of_pycol_class_values\')\n    within=X[y==0].var(axis=0)+X[y==1].var(axis=0)\n    different=X[y==0].mean(axis=0)!=X[y==1].mean(axis=0)\n    active=np.ptp(X,axis=0)>0\n    with warnings.catch_warnings():\n        warnings.simplefilter(\'ignore\',RuntimeWarning)\n        f1=np.asarray(c.F1(),dtype=float)\n    ftable=pd.DataFrame({\'feature_index\':np.arange(X.shape[1]),\'pycol_F1_overlap\':f1,\n        \'nonconstant\':active,\'zero_within_variance_different_means\':(within==0)&different})\n    good=active & ~((within==0)&different)\n    # Never silently trust upstream F1\'s inf->0 conversion on perfect constant-per-class features.\n    invalid=bool(((within==0)&different).any()) or not active.any()\n    for metric,fun in [(\'Fisher_overlap_min\',np.min),(\'Fisher_overlap_mean\',np.mean)]:\n        add(metric,\'overall\',np.nan if invalid else fun(f1[good]),\n            implementation=\'summary_of_pycol_F1_nonconstant_features\',\n            status=\'undefined: degenerate Fisher denominator\' if invalid else \'ok\')\n    D=c.dist_matrix.copy(); np.fill_diagonal(D,np.inf)\n    nearest=np.argmin(D,axis=1); nd=D[np.arange(len(y)),nearest]\n    ties=np.sum(np.isclose(D,nd[:,None],rtol=1e-12,atol=1e-12),axis=1)>1\n    detail=frame[[\'sample_id\',\'patient_id\',\'label\']].copy()\n    detail[\'nearest_sample_id\']=frame.sample_id.to_numpy()[nearest]\n    detail[\'nearest_same_patient\']=groups==groups[nearest]\n    detail[\'nearest_distance_tie\']=ties\n    if patient_checks:\n        D[groups[:,None]==groups[None,:]]=np.inf\n        ix=np.argmin(D,axis=1); finite=np.isfinite(D[np.arange(len(y)),ix])\n        error=(y!=y[ix]).astype(float); error[~finite]=np.nan\n        same=y[:,None]==y[None,:]\n        within_dist=np.min(np.where(same,D,np.inf),axis=1)\n        between_dist=np.min(np.where(~same,D,np.inf),axis=1)\n        for scope,mask in [(\'overall\',np.ones(len(y),bool)),(\'class_0\',y==0),(\'class_1\',y==1)]:\n            ok=finite[mask].all()\n            add(\'N3_other_patient\',scope,np.mean(error[mask]) if ok else np.nan,\n                \'custom_patient_excluded_1NN\',\'ok\' if ok else \'undefined: no eligible neighbour\')\n            a=within_dist[mask]; b=between_dist[mask]\n            ok=np.isfinite(a).all() and np.isfinite(b).all() and b.sum()>0\n            add(\'N2_other_patient\',scope,a.sum()/(a.sum()+b.sum()) if ok else np.nan,\n                \'custom_patient_excluded_distance_ratio\',\'ok\' if ok else \'undefined: missing eligible class neighbour\')\n        for metric in (\'N2_other_patient\',\'N3_other_patient\'):\n            vals=[r[\'value\'] for r in rows if r[\'metric\']==metric and r[\'scope\'] in (\'class_0\',\'class_1\')]\n            add(metric,\'macro\',np.mean(vals),\'mean_of_custom_class_values\',\n                \'ok\' if np.isfinite(vals).all() else \'undefined: class estimate unavailable\')\n        detail[\'other_patient_nearest_sample_id\']=frame.sample_id.to_numpy()[ix]\n        detail[\'other_patient_1NN_error\']=error\n    audit=dict(images=len(y),features=X.shape[1],constant_features=int((~active).sum()),\n        zero_distance_pairs=int((off==0).sum()),near_zero_pairs_MST=int((off<=1e-8).sum()),\n        nearest_tie_images=int(ties.sum()),nearest_same_patient_fraction=float(detail.nearest_same_patient.mean()),\n        mixed_label_patients=int((frame.groupby(\'patient_id\').label.nunique()>1).sum()))\n    del c,D; gc.collect()\n    return pd.DataFrame(rows),ftable,detail,audit\n\ndef balanced_patient_plans(frames,repeats=20,seed=42,cap_per_class=250):\n    """One random image/patient, then equal class counts shared across datasets/repeats.\n    Mixed-label patients may represent a different label in different repeats.\n    This changes class prevalence and cohort: sensitivity only, not a CI.\n    """\n    candidates={}; counts=[]\n    for ds,f in frames.items():\n        for repeat in range(repeats):\n            rng=np.random.default_rng(np.random.SeedSequence([seed,repeat,int(hashlib.sha256(ds.encode()).hexdigest()[:8],16)]))\n            ix=np.array([rng.choice(np.asarray(g,dtype=int)) for g in f.groupby(\'patient_id\',sort=True).indices.values()])\n            candidates[(ds,repeat)]=ix\n            counts.extend([(f.iloc[ix].label==label).sum() for label in (0,1)])\n    k=int(min(cap_per_class,min(counts)))\n    if k<2: raise RuntimeError(\'Not enough distinct patients per class for balanced sensitivity\')\n    plans={}\n    for (ds,repeat),ix in candidates.items():\n        rng=np.random.default_rng(seed+10000+repeat)\n        chosen=np.concatenate([rng.choice(ix[frames[ds].iloc[ix].label.to_numpy()==label],k,replace=False) for label in (0,1)])\n        plans[(ds,repeat)]=np.sort(chosen)\n        assert frames[ds].iloc[chosen].patient_id.is_unique\n    return k,plans\n'
SAMPLER_SOURCE='import hashlib\nimport numpy as np\nimport pandas as pd\n\ndef patient_pools(frame):\n    """Index positions, never CSV index labels. Preserve each image\'s true label."""\n    if frame[[\'sample_id\',\'patient_id\',\'label\']].isna().any().any():\n        raise ValueError(\'Missing identity/label\')\n    if not frame.sample_id.is_unique or not frame.label.isin([0,1]).all():\n        raise ValueError(\'Invalid IDs/labels\')\n    pools={0:[],1:[],2:[]}; images={}\n    for pid,positions in frame.groupby(\'patient_id\',sort=True).indices.items():\n        positions=np.asarray(positions,dtype=int)\n        labels=frame.iloc[positions].label.to_numpy(dtype=int)\n        available=set(labels)\n        key=2 if available=={0,1} else next(iter(available))\n        pools[key].append(str(pid))\n        images[str(pid)]={lab:positions[labels==lab] for lab in available}\n    a,b,m=map(lambda k:len(pools[k]),[0,1,2])\n    capacity=min(a+m,b+m,(a+b+m)//2)\n    return pools,images,dict(negative_only=a,positive_only=b,mixed=m,\n        negative_eligible=a+m,positive_eligible=b+m,patients=a+b+m,max_balanced_per_class=capacity)\n\ndef label_aware_plan(frame,k,rng):\n    pools,images,counts=patient_pools(frame)\n    if not 2<=k<=counts[\'max_balanced_per_class\']:raise ValueError(\'Infeasible K\')\n    a,b,m=counts[\'negative_only\'],counts[\'positive_only\'],counts[\'mixed\']\n    # Sample uniformly over feasible mixed-patient COUNT pairs, then over identities.\n    # Not uniform over all feasible subsets; this selection distribution is explicit.\n    pairs=[(x0,x1) for x0 in range(max(0,k-a),min(k,m)+1)\n           for x1 in range(max(0,k-b),min(k,m-x0)+1)]\n    x0,x1=pairs[int(rng.integers(len(pairs)))]\n    mixed=rng.permutation(pools[2]).tolist()\n    assignments=[(pid,0) for pid in rng.choice(pools[0],k-x0,replace=False).tolist()+mixed[:x0]]\n    assignments +=[(pid,1) for pid in rng.choice(pools[1],k-x1,replace=False).tolist()+mixed[x0:x0+x1]]\n    selected=np.sort([int(rng.choice(images[pid][label])) for pid,label in assignments])\n    subset=frame.iloc[selected]\n    assert subset.patient_id.is_unique and subset.sample_id.is_unique\n    assert subset.label.value_counts().to_dict()=={0:k,1:k}\n    return selected,dict(mixed_assigned_negative=x0,mixed_assigned_positive=x1,\n        negative_only_selected=k-x0,positive_only_selected=k-x1)\n\ndef build_plans(frames,repeats=20,seed=42,cap=250):\n    capacities=pd.DataFrame([dict(dataset=ds,**patient_pools(f)[2]) for ds,f in frames.items()])\n    k=int(min(cap,capacities.max_balanced_per_class.min()))\n    if k<2:raise ValueError(\'Insufficient patient groups for balanced comparison\')\n    plans={}; audit=[]\n    for ds,f in frames.items():\n        tag=int(hashlib.sha256(ds.encode()).hexdigest()[:8],16)\n        for repeat in range(repeats):\n            rng=np.random.default_rng(np.random.SeedSequence([seed,repeat,tag]))\n            ix,allocation=label_aware_plan(f,k,rng)\n            plans[(ds,repeat)]=ix\n            audit.append(dict(dataset=ds,repeat=repeat,k_per_class=k,images=2*k,\n                unique_patients=int(f.iloc[ix].patient_id.nunique()),**allocation))\n    return k,plans,capacities,pd.DataFrame(audit)\n'
EXPECTED_SOURCE_HASHES={'protocol.json': '82f254017fb3ead678f233a1c2cd630498760271ae0f974652e840338b4b2bdc', 'self_test.json': '13f894722cccee5b8f1b7ee3ed7b78c06a5eac036252642f04772dc35b19823e', 'cohort_audit.csv': '852d32b05acd22d6661ede69a94ac5f969684caa8dcaa2775e9c82d69cf48d74', 'b07_pycol_engine.py': '1a8fc0ca06b45e00417c6c388c7882d548a473a5f1e50d0ae40bc746edaefae6', 'full_train_metrics.csv': '9c641e22458439c9898d83ae6a5553246fd6084ba2eefa93502e205f8166485b', 'distance_audit.csv': 'a18513eaf15d3d20e25cc6e20694607ed9c8393b79ce34043e7a35008096e3fa', 'subset_metrics_all_repeats.csv': '5fdfb7a9889d1a1113dbeaaabb1b302f63d2a105ef697ee921a3dd506bacd30d', 'subset_membership.csv': '928f4dc5f78eb414315a07f3aebbdf7a39f705a32e9a6a1fa0ed260323fcaf79', 'subset_summary.csv': '75dcafacfd994bea756b84db0e9c6f5cb85942d2d3a91a210205e97808b0b802', 'subset_design.json': '703a088f69ae01868a7a3f8493f6bf52a0567fdf483e1bd5c4cbd63bb196e9d3', 'pycol_full_train.png': '63e771684ff01d7c718f4274cab6b184dc271bfe4bb207f2a199f4b24b04b51c', 'pycol_subset_sensitivity.png': '4dd4bf60243fa2c198a083248d139b375b1e7b406f3ad679be6c249d88aa873f', 'README.txt': '26e51240d1bc72385e4bc13a8ff9e2443ec063a5981f4fb3a4f6f33d7f7a7f30', 'full_train/cbis_ddsm/metrics.csv': 'b01f936b1706344ee071fbe619eb6547a30fb3032f82716373c5aa16797c8e01', 'full_train/cbis_ddsm/fisher_per_feature.csv': 'b693e3898b3225ad19c53d44bba693a8a959e48df3ff33120c36a5cafcabefa2', 'full_train/cbis_ddsm/neighbour_audit.csv': '93a7c86e55bc86fe61a659a1c9e1906cd329c40f7ef3680894d6c9ccb07884a4', 'full_train/cbis_ddsm/distance_audit.json': '042322adfe02f3ceaef9961a6b9ffaab3c025683f49e1e7828005bdd536f7f41', 'full_train/cmmd/metrics.csv': '14ea37f748ec08a27b15287787fc5d2fe6b95e751ff0af838e3ae7628967cb55', 'full_train/cmmd/fisher_per_feature.csv': '58685aea4ade1c35890b90400ebf07f125863f3f77289c927df861de2d546e33', 'full_train/cmmd/neighbour_audit.csv': 'dcea69654138afd23c917025d8332719c94fd611919f337c14b0ae6b3bc63774', 'full_train/cmmd/distance_audit.json': '5585f98ce9bee37619bac1ab2b7266c57713ecfd923f567e1e0e89d21166cc5d', 'full_train/inbreast/metrics.csv': '7d272a37389a72216210b5c71bc0099a26cf744fb5de9949d24f95fd8694d096', 'full_train/inbreast/fisher_per_feature.csv': 'b0a0af52003012b2a5a69d29fd1ed7ebd4ef0a3ea92243dc927f616faf6c7582', 'full_train/inbreast/neighbour_audit.csv': '7f4104649e466ab9185e798a36e7088d93f23db05fb940187ff72910b15a9ed1', 'full_train/inbreast/distance_audit.json': '802fa6d541af407932aa82560998c2282acf217c8eb4990094e43cfbd2c72ace', 'full_train/mias/metrics.csv': 'c17601ccd842e43f3b595df30b6d1091107f750d92d70a6ff808a3d01ef4dd1d', 'full_train/mias/fisher_per_feature.csv': '81c152ec1cdb53bc0ce57c1c99809ed47fd8168bf18d4b268b3d723de37c90ba', 'full_train/mias/neighbour_audit.csv': '24c290fd10977dda3d62a78fac60c74e2904b4b5f25ef7a84ae3f673c830aab2', 'full_train/mias/distance_audit.json': '267c75a291d3b3b3d7d82607474b560339f84634264efbf0b0b75477058d9c46'}
exec(compile(ENGINE_SOURCE,'b07_original_engine.py','exec'),globals())
exec(compile(SAMPLER_SOURCE,'b07a_label_aware_sampler.py','exec'),globals())
source_protocol=read_json(SOURCE/'protocol.json')
if object_sha(source_protocol)[:16]!=SOURCE_ID: raise RuntimeError('Protokol B07 berubah')
if file_sha(SOURCE/'protocol.json')!=EXPECTED_SOURCE_HASHES['protocol.json']: raise RuntimeError('B07 bukan versi yang diaudit')
if metadata.version('pycol-complexity')!='1.0.4': raise RuntimeError('Pycol version mismatch')
pycol_path=Path(inspect.getfile(pycol_module))
if file_sha(pycol_path)!=source_protocol['pycol_source_sha256']: raise RuntimeError('Source Pycol berubah')
for name in ['subset_summary.csv','full_train_metrics.csv','cohort_audit.csv','distance_audit.csv','subset_design.json']:
    if file_sha(SOURCE/name)!=EXPECTED_SOURCE_HASHES[name]: raise RuntimeError(f'Hasil B07 berubah: {name}')
SELF_TEST=self_test()
print(SELF_TEST)


## Cell 3 — Membaca cache fitur dan memvalidasi label
Hash fitur harus cocok dengan COMPLETE.json. Urutan sample_id, patient_id, dan label dicocokkan dengan CSV train asli yang hash-nya telah dibekukan. Tidak menggunakan TensorFlow atau citra.

In [ ]:
registry=read_json(REGISTRY_PATH)
FRAMES={};FEATURES={};cache_hashes={}
for ds in DATASETS:
    print('Memuat cache:',ds,flush=True)
    provenance=source_protocol['provenance'][ds]
    if file_sha(REGISTRY_PATH)!=provenance['registry_sha256']: raise RuntimeError('Registry B04 berubah')
    entry=registry[ds]['full_512']
    if entry['status']!='READY': raise RuntimeError('Cache B04 tidak READY')
    train_path=Path(entry['path'])/'train.csv'
    if file_sha(train_path)!=provenance['ready']['file_sha256']['train.csv']: raise RuntimeError('CSV train berubah')
    original=pd.read_csv(train_path,dtype={'sample_id':str,'patient_id':str}).sort_values('sample_id').reset_index(drop=True)
    if original[['sample_id','patient_id','label']].isna().any().any() or not original.sample_id.is_unique:
        raise ValueError('Invalid source IDs')
    if set(original.split)!={'train'} or set(original.label)!={0,1}: raise ValueError('Invalid split/labels')
    cache=SOURCE/'feature_cache'/ds
    done=read_json(cache/'COMPLETE.json')
    manifest=done['manifest']
    if manifest['protocol_id']!=SOURCE_ID or manifest['weights_sha256']!=source_protocol['weights_sha256']:
        raise RuntimeError('Cache feature berasal dari protokol/model berbeda')
    f=pd.read_csv(cache/'feature_index.csv',dtype={'sample_id':str,'patient_id':str})
    if f.sample_id.tolist()!=manifest['ordered_ids'] or f.sample_id.tolist()!=original.sample_id.tolist():
        raise RuntimeError('Feature row ordering mismatch')
    for col in ['sample_id','patient_id','label']:
        if not np.array_equal(f[col].to_numpy(),original[col].to_numpy()): raise RuntimeError(f'Label/ID mismatch: {col}')
    if object_sha(f.sample_id.tolist())!=provenance['ordered_train_ids_sha256']: raise RuntimeError('ID hash mismatch')
    feature_path=cache/'features.npy'
    if file_sha(feature_path)!=done['features_sha256']: raise RuntimeError('Feature cache hash mismatch')
    X=np.load(feature_path,mmap_mode='r',allow_pickle=False)
    if list(X.shape)!=done['shape'] or X.shape!=(len(f),2048) or not np.isfinite(X).all():
        raise ValueError('Invalid feature matrix')
    FRAMES[ds]=f;FEATURES[ds]=X
    cache_hashes[ds]=dict(features_sha256=done['features_sha256'],index_sha256=file_sha(cache/'feature_index.csv'))
K,PLANS,CAPACITY,ALLOCATION=build_plans(FRAMES,REPEATS,SUBSET_SEED,CAP_PER_CLASS)
display(CAPACITY)
print(f'Setiap subset: {K} negatif + {K} positif dari {2*K} pasien berbeda.')
environment={k:metadata.version(k) for k in ['numpy','pandas','scipy','scikit-learn','pycol-complexity']}
environment['python']=platform.python_version()
PROTOCOL=dict(version='b07a_label_aware_v1',source_id=SOURCE_ID,source_protocol_sha256=file_sha(SOURCE/'protocol.json'),
    datasets=DATASETS,repeats=REPEATS,subset_seed=SUBSET_SEED,cap_per_class=CAP_PER_CLASS,k_per_class=K,
    sampler='uniform feasible mixed-patient count pairs; random patients without replacement; random image of assigned label',
    scope='train only; same frozen ResNet50 features and Pycol measures as B07',
    engine_sha256=hashlib.sha256(ENGINE_SOURCE.encode()).hexdigest(),
    sampler_sha256=hashlib.sha256(SAMPLER_SOURCE.encode()).hexdigest(),cache_hashes=cache_hashes,environment=environment)
EXPERIMENT_ID=object_sha(PROTOCOL)[:16]
stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUT=OUTPUT_ROOT/EXPERIMENT_ID/stamp;OUT.mkdir(parents=True,exist_ok=False)
save_json(OUT/'protocol.json',PROTOCOL);save_json(OUT/'source_protocol.json',source_protocol)
save_json(OUT/'self_test.json',SELF_TEST)
CAPACITY.to_csv(OUT/'patient_capacity.csv',index=False)
ALLOCATION.to_csv(OUT/'allocation_audit.csv',index=False)
(OUT/'b07_original_engine.py').write_text(ENGINE_SOURCE)
(OUT/'b07a_sampler.py').write_text(SAMPLER_SOURCE)
print('Output baru:',OUT)


## Cell 4 — Hitung ulang subset dan simpan hasil
Setiap subset wajib unik pada sample_id dan patient_id, serta memiliki K citra untuk masing-masing kelas. File keanggotaan memungkinkan pemeriksaan ulang.

Jika sebuah metrik tidak tersedia pada salah satu pengulangan, ringkasan metrik itu ditandai tidak lengkap. Mean dari sebagian pengulangan yang kebetulan valid tidak ditampilkan sebagai hasil lengkap.

In [ ]:
rows=[];members=[]
for (ds,repeat),ix in PLANS.items():
    f=FRAMES[ds].iloc[ix].reset_index(drop=True)
    assert f.patient_id.is_unique and f.sample_id.is_unique
    assert f.label.value_counts().to_dict()=={0:K,1:K}
    t,_,_,_=measure(np.asarray(FEATURES[ds][ix]),f,ds,
        scenario='label_aware_balanced_one_image_per_patient',repeat=repeat,patient_checks=False)
    rows.append(t)
    members.append(f[['sample_id','patient_id','label']].assign(dataset=ds,repeat=repeat))
    if repeat%5==0:print(ds,repeat+1,'/',REPEATS,flush=True)
METRICS=pd.concat(rows,ignore_index=True)
MEMBERS=pd.concat(members,ignore_index=True)
SUMMARY=METRICS.groupby(['dataset','metric','scope'],as_index=False).agg(
    mean=('value','mean'),sample_sd=('value','std'),minimum=('value','min'),maximum=('value','max'),
    valid_repeats=('value','count'),requested_repeats=('value','size'))
SUMMARY['images_per_class']=K
SUMMARY['status']=np.where(SUMMARY.valid_repeats==REPEATS,'complete','incomplete: inspect metric status')
SUMMARY.loc[SUMMARY.valid_repeats!=REPEATS,['mean','sample_sd','minimum','maximum']]=np.nan
METRICS.to_csv(OUT/'subset_metrics_all_repeats.csv',index=False)
MEMBERS.to_csv(OUT/'subset_membership.csv',index=False)
SUMMARY.to_csv(OUT/'subset_summary.csv',index=False)
coverage=MEMBERS.groupby(['dataset','label'],as_index=False).agg(
    distinct_patients_across_repeats=('patient_id','nunique'),distinct_images_across_repeats=('sample_id','nunique'))
coverage.to_csv(OUT/'sampling_coverage.csv',index=False)
display(SUMMARY[(SUMMARY.scope=='overall') & SUMMARY.metric.isin(['N1','N2','N3'])].round(4))
display(coverage)


## Cell 5 — Perbandingan B07 lama dan B07A
Tabel memisahkan angka lama dan baru. Delta adalah perubahan mean deskriptif, **bukan peningkatan performa model**. Sampler dan ukuran subset sama-sama berubah. Error bar adalah sample SD antarsubset, bukan CI.

N1/N2/N3 makin tinggi berarti lebih sulit menurut ukuran itu. `Fisher_overlap_min/mean` adalah ringkasan Fisher per fitur, bukan F1-score klasifikasi. Jangan menafsirkan perubahan Fisher minimum sebagai pembelajaran model: ukuran subset juga memengaruhi statistik fitur ekstrem.

In [ ]:
old=pd.read_csv(SOURCE/'subset_summary.csv')
COMPARISON=old.merge(SUMMARY,on=['dataset','metric','scope'],suffixes=('_B07','_B07A'),validate='one_to_one')
COMPARISON['delta_mean_B07A_minus_B07']=COMPARISON['mean_B07A']-COMPARISON['mean_B07']
COMPARISON.to_csv(OUT/'comparison_old_new.csv',index=False)
display(COMPARISON[(COMPARISON.scope=='overall')&COMPARISON.metric.isin(['N1','N2','N3'])][
    ['dataset','metric','images_per_class_B07','mean_B07','sample_sd_B07','images_per_class_B07A','mean_B07A','sample_sd_B07A']].round(4))
fig,axes=plt.subplots(1,3,figsize=(14,4),constrained_layout=True)
for ax,metric in zip(axes,['N1','N2','N3']):
    t=SUMMARY[(SUMMARY.metric==metric)&(SUMMARY.scope=='overall')].set_index('dataset').reindex(DATASETS)
    good=t['mean'].notna()&t.sample_sd.notna()
    if good.any(): ax.bar(t.index[good],t.loc[good,'mean'],yerr=t.loc[good,'sample_sd'],capsize=4,color='#356D9B')
    else:ax.text(.5,.5,'Unavailable: inspect status',transform=ax.transAxes,ha='center')
    ax.set_title(f'{metric}: {2*K} images, unique patients');ax.set_ylim(0,1)
    ax.set_ylabel('Mean ± sample SD (not CI)');ax.tick_params(axis='x',rotation=20)
fig.savefig(OUT/'label_aware_subset_metrics.png',dpi=180);plt.show();plt.close(fig)
README="""B07A: label-aware balanced subset sensitivity using B07 cached features.
No training, no feature extraction, no test access. Original B07 files are read-only.
Uniform sampling of feasible mixed-patient COUNT pairs is not uniform over all feasible subsets.
Patients may appear again in another repeat; each individual subset has no repeated patient.
INbreast at K=21 uses all 21 positive-eligible patients each time; repeats are dependent.
Mean/SD are sensitivity descriptors, not confidence intervals or training-seed statistics.
Old vs new differs in both sampler and subset size. Do not claim a pure sample-size effect.
Retain full-train results as separate context; target labels differ among datasets.
Patient identities/grouping are inherited from B07 and not clinically reverified here.
Fisher overlap is not classification F1. No composite complexity ranking is calculated.
Reference: https://doi.org/10.1016/j.neucom.2025.130311
"""
(OUT/'README.txt').write_text(README)


## Cell 6 — ZIP laporan
Kirim ZIP B07A setelah selesai. Fitur dan citra tidak dimasukkan dalam ZIP; cache B07 tetap di Drive.

In [ ]:
for name in ['subset_summary.csv','full_train_metrics.csv','cohort_audit.csv','distance_audit.csv','subset_design.json']:
    (OUT/('source_B07_'+name)).write_bytes((SOURCE/name).read_bytes())
report_files=sorted(p for p in OUT.iterdir() if p.is_file())
save_json(OUT/'report_manifest.json',dict(protocol_id=EXPERIMENT_ID,source_id=SOURCE_ID,created_utc=stamp,
    datasets=DATASETS,k_per_class=K,repeats=REPEATS,metric_rows=len(METRICS),
    undefined_rows=int((METRICS.status!='ok').sum()),
    files={p.name:file_sha(p) for p in report_files}))
report_dir=OUTPUT_ROOT/'reports';report_dir.mkdir(parents=True,exist_ok=True)
REPORT=report_dir/f'B07A_Pycol_{EXPERIMENT_ID}_{stamp}_report.zip'
with zipfile.ZipFile(REPORT,'x',compression=zipfile.ZIP_DEFLATED) as archive:
    for p in report_files+[OUT/'report_manifest.json']:archive.write(p,arcname=p.name)
print('Selesai:',REPORT)
if DOWNLOAD_REPORT:files.download(str(REPORT))
